In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

In [ ]:
# import pandas as pd

# # =========================
# # LOAD PREDICTIONS
# # =========================

# lstm_sub = pd.read_csv(
#     "/kaggle/input/notebooks/vishalsharmaaaaaa/03-lstm-ipynb/submission.csv"
# )

# tfidf_sub = pd.read_csv(
#     "/kaggle/input/notebooks/vishalsharmaaaaaa/02-tfidf/submission.csv"
# )

# deberta_sub = pd.read_csv(
#     "/kaggle/input/notebooks/vishalsharmaaaaaa/dl-23f2004341-notebook-t22026/submission.csv"
# )

# print("LSTM:", lstm_sub.shape)
# print("TFIDF:", tfidf_sub.shape)
# print("DeBERTa:", deberta_sub.shape)

# # =========================
# # WEIGHTED ENSEMBLE
# # =========================

# final_predictions = []

# for i in range(len(deberta_sub)):

#     lstm_pred = str(
#         lstm_sub.iloc[i]["Prediction"]
#     ).split()

#     tfidf_pred = str(
#         tfidf_sub.iloc[i]["Prediction"]
#     ).split()

#     deberta_pred = str(
#         deberta_sub.iloc[i]["Prediction"]
#     ).split()

#     scores = {
#         "A":0,
#         "B":0,
#         "C":0,
#         "D":0,
#         "E":0
#     }

#     # DeBERTa (70%)

#     for rank, option in enumerate(deberta_pred):

#         scores[option] += (
#             0.70 * (3-rank)
#         )

#     # TFIDF (20%)

#     for rank, option in enumerate(tfidf_pred):

#         scores[option] += (
#             0.20 * (3-rank)
#         )

#     # LSTM (10%)

#     for rank, option in enumerate(lstm_pred):

#         scores[option] += (
#             0.10 * (3-rank)
#         )

#     ranked = sorted(
#         scores.items(),
#         key=lambda x: x[1],
#         reverse=True
#     )

#     prediction = " ".join(
#         [x[0] for x in ranked[:3]]
#     )

#     final_predictions.append(
#         prediction
#     )

# # =========================
# # SUBMISSION
# # =========================

# submission = pd.DataFrame({

#     "ID":
#     deberta_sub["ID"],

#     "Prediction":
#     final_predictions
# })

# submission.to_csv(
#     "/kaggle/working/submission.csv",
#     index=False
# )

# print("\nENSEMBLE CREATED")
# print(submission.head())

# print(
#     "\nSaved File:",
#     "/kaggle/working/submission.csv"
# )

In [ ]:
import pandas as pd
import numpy as np

# =====================================================
# LOAD TEST FILE
# =====================================================

test = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
)

# =====================================================
# LOAD SUBMISSIONS
# =====================================================

tfidf_sub = pd.read_csv(
    "/kaggle/input/notebooks/vishalsharmaaaaaa/02-tfidf/submission.csv"
)

lstm_sub = pd.read_csv(
    "/kaggle/input/notebooks/vishalsharmaaaaaa/03-lstm-ipynb/submission.csv"
)

deberta_sub = pd.read_csv(
    "/kaggle/input/notebooks/vishalsharmaaaaaa/dl-23f2004341-notebook-t22026/submission.csv"
)

print("Loaded:")
print("TFIDF:", len(tfidf_sub))
print("LSTM:", len(lstm_sub))
print("DeBERTa:", len(deberta_sub))

# =====================================================
# DETECT COLUMN NAMES
# =====================================================

def get_prediction_column(df):
    for col in df.columns:
        if col.lower() == "prediction":
            return col
    raise ValueError(f"Prediction column not found. Columns = {df.columns}")

tfidf_col = get_prediction_column(tfidf_sub)
lstm_col = get_prediction_column(lstm_sub)
deberta_col = get_prediction_column(deberta_sub)

# =====================================================
# WEIGHTS
# =====================================================

TFIDF_WEIGHT = 0.10
LSTM_WEIGHT = 0.00
DEBERTA_WEIGHT = 0.90
# =====================================================
# ENSEMBLE
# =====================================================

final_predictions = []

for i in range(len(test)):

    tfidf_pred = str(tfidf_sub.iloc[i][tfidf_col]).split()
    lstm_pred = str(lstm_sub.iloc[i][lstm_col]).split()
    deberta_pred = str(deberta_sub.iloc[i][deberta_col]).split()

    option_scores = {
        "A": 0.0,
        "B": 0.0,
        "C": 0.0,
        "D": 0.0,
        "E": 0.0
    }

    # TFIDF
    for rank, option in enumerate(tfidf_pred):
        option_scores[option] += TFIDF_WEIGHT * (3 - rank)

    # LSTM
    for rank, option in enumerate(lstm_pred):
        option_scores[option] += LSTM_WEIGHT * (3 - rank)

    # DeBERTa
    for rank, option in enumerate(deberta_pred):
        option_scores[option] += DEBERTA_WEIGHT * (3 - rank)

    top3 = sorted(
        option_scores.keys(),
        key=lambda x: option_scores[x],
        reverse=True
    )[:3]

    final_predictions.append(" ".join(top3))

# =====================================================
# SUBMISSION
# =====================================================

submission = pd.DataFrame({
    "id": test["id"],
    "prediction": final_predictions
})

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print("\nSubmission Saved Successfully")
print(submission.head())

print("\nFile Location:")
print("/kaggle/working/submission.csv")